# T05 - Thiết lập pipeline và huấn luyện Linear Regression
Đầu vào: `data_train.csv` và `data_test.csv` (T04) đã xuất trong thư mục `data_processed`. Hai file này đã One-Hot, Scale và có cột giá `price_million`.

**T05:** Huấn luyện mô hình cơ bản (Linear Regression) và lưu lại. 

Cài một lần: `pip install pandas numpy scikit-learn joblib ipykernel`

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression

candidates = [Path("data_processed"), Path("../../Feature Engineering/data_processed")]
DATA_DIR = next((p for p in candidates if (p / "data_train.csv").exists()), None)
if DATA_DIR is None:
    DATA_DIR = next(Path.cwd().parent.rglob("data_train.csv")).parent
print("Đọc dữ liệu từ:", DATA_DIR.resolve())

Đọc dữ liệu từ: C:\Users\ADMIN\Documents\GitHub\USED-CAR-PRICE-PREDICTION-ADY201m\Feature Engineering\data_processed


In [30]:
TARGET = "price_million"
train = pd.read_csv(DATA_DIR / "data_train.csv")
test = pd.read_csv(DATA_DIR / "data_test.csv")

X_train, y_train = train.drop(columns=TARGET), train[TARGET]
X_test, y_test = test.drop(columns=TARGET), test[TARGET]

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Có ô trống không:", train.isna().sum().sum() + test.isna().sum().sum())

Train: (9995, 134) | Test: (2499, 134)
Có ô trống không: 0


## 1. Huấn luyện
Dự đoán `log(giá)` rồi đổi ngược về giá (`TransformedTargetRegressor`). Cách này không cho ra giá âm, và hợp với giá xe vì xe cũ mất giá theo phần trăm.

In [31]:
reg = TransformedTargetRegressor(LinearRegression(), func=np.log, inverse_func=np.exp)
reg.fit(X_train, y_train)
print("Đã huấn luyện xong trên", len(X_train), "dòng,", X_train.shape[1], "đặc trưng")

Đã huấn luyện xong trên 9995 dòng, 134 đặc trưng


## 2. Lưu mô hình
- `linear_regressor.pkl`: mô hình chạy trực tiếp trên `data_test.csv` (đã mã hóa). **T07 dùng file này** để tính chỉ số.
- `linear_model.pkl`: pipeline đầy đủ (bộ mã hóa của Nhân + mô hình) cho T09 demo.

In [ ]:
joblib.dump(reg, "linear_regressor.pkl")
print("Đã lưu linear_regressor.pkl")

prep_path = DATA_DIR / "preprocessor.joblib"
if prep_path.exists():
    full_model = Pipeline([("prep", joblib.load(prep_path)), ("reg", reg)])
    joblib.dump(full_model, "linear_model.pkl")
    print("Đã lưu linear_model.pkl (pipeline đầy đủ)")
else:
    print("Chưa có preprocessor.joblib -> chưa tạo linear_model.pkl")

Đã lưu linear_regressor.pkl
Đã lưu linear_model.pkl (pipeline đầy đủ)
